In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ChicagoCrimes")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

path = "Crimes_-_2001_to_Present.csv"

print("loading csv")

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)

print("removing duplicates")

df = df.dropDuplicates()

print("fixing dates")

df = df.withColumn(
    "date_ts",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

print("removing wrong dates")

df = df.filter(col("date_ts").isNotNull())

df = df.filter(
    (year(col("date_ts")) >= 2001) &
    (year(col("date_ts")) <= year(current_date()))
)

print("removing missing values")

df = df.dropna(subset=[
    "ID",
    "Case Number",
    "Primary Type",
    "Location Description"
])

print("filtering coordinates")

df = df.filter(
    (
        col("Latitude").isNull() |
        col("Latitude").between(41.0, 43.0)
    ) &
    (
        col("Longitude").isNull() |
        col("Longitude").between(-88.5, -87.0)
    )
)

print("creating time columns")

df = df.withColumn("year", year(col("date_ts")))
df = df.withColumn("month", month(col("date_ts")))
df = df.withColumn("hour", hour(col("date_ts")))
df = df.withColumn("day_of_week", date_format(col("date_ts"), "E"))

def day_period(hour):
    if hour is None:
        return "unknown"
    elif 0 <= hour < 6:
        return "night"
    elif 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "day"
    else:
        return "evening"

day_period_udf = udf(day_period, StringType())

print("adding udf column")

df = df.withColumn(
    "day_period",
    day_period_udf(col("hour"))
)

print("cache dataframe")

df = df.cache()

df.count()

print("creating small table")

locations = spark.createDataFrame([
    ("STREET", "public"),
    ("SIDEWALK", "public"),
    ("RESIDENCE", "private"),
    ("APARTMENT", "private"),
    ("SCHOOL", "school"),
    ("CTA TRAIN", "transport"),
    ("CTA BUS", "transport")
], ["Location Description", "location_group"])

print("broadcast join")

df = df.join(
    broadcast(locations),
    on="Location Description",
    how="left"
)

df = df.withColumn(
    "location_group",
    coalesce(col("location_group"), lit("other"))
)

print("saving parquet")

(
    df.repartition("year")
    .write
    .mode("overwrite")
    .partitionBy("year")
    .parquet("chicago_crimes_parquet")
)

print("reading parquet")

crimes = spark.read.parquet("chicago_crimes_parquet")

print("\nanalysis 1")

query1 = (
    crimes
    .groupBy("Primary Type")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

query1.explain(True)

query1.show(20, False)

print("\nanalysis 2")

query2 = (
    crimes
    .groupBy("Location Description")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

query2.explain(True)

query2.show(20, False)

print("\nanalysis 3")

query3 = (
    crimes
    .groupBy("location_group")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

query3.explain(True)

query3.show(20, False)

print("\nanalysis 4")

query4 = (
    crimes
    .groupBy("year", "month")
    .agg(count("*").alias("count"))
    .orderBy("year", "month")
)

query4.explain(True)

query4.show(50, False)

print("\nanalysis 5")

query5 = (
    crimes
    .groupBy("day_period")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

query5.explain(True)

query5.show(False)

print("\nanalysis 6")

query6 = (
    crimes
    .groupBy("day_of_week")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
)

query6.explain(True)

query6.show(False)

print("\nanalysis 7")

window_spec = (
    Window
    .partitionBy("day_period")
    .orderBy(desc("count"))
)

query7 = (
    crimes
    .groupBy("day_period", "Primary Type")
    .agg(count("*").alias("count"))
)

query7 = query7.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

query7 = query7.filter(col("rank") <= 5)

query7 = query7.orderBy("day_period", "rank")

query7.explain(True)

query7.show(100, False)

spark.stop()

![opis](zdj1.png)

![opis](zdj2.png)

![opis](zdj3.png)